# Notebook Setup

In [1]:
# functionality check
import socket
import sys
from pathlib import Path

print("Container hostname:", socket.gethostname())
print("Python executable:", sys.executable)
print("Working directory:", Path.cwd())

Container hostname: 2dfbb1ac98d0
Python executable: /usr/local/bin/python
Working directory: /app/evaluation_results


In [2]:
# imports
from pathlib import Path
import json

import pandas as pd
from IPython.display import Markdown, display

# Notebook and JSON files are in the same directory.
DATA_DIR = Path.cwd()

# Generated CSV files are saved in this subfolder.
OUTPUT_DIR = DATA_DIR / "evaluation_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# Evaluation – Power Analysis

In [3]:
# =============================================================================
# PILOT-STUDY POWER ANALYSIS
# Repeated-measures ANOVA approximation for the Friedman tests
# =============================================================================

import math

import pandas as pd
import pingouin as pg


# Study design
N_ALGORITHMS = 8
PILOT_N = 9
TARGET_POWER = 0.80

# Four planned outcomes:
# relevance, viewing intention, novelty, and Top-5 performance
ALPHA = 0.05 / 4

# Planning assumptions
AVERAGE_CORRELATION = 0.30
EPSILON = 0.50

# Conventional Cohen's f scenarios
effect_sizes = {
    "Small": 0.10,
    "Medium": 0.25,
    "Large": 0.40,
}

power_rows = []

for effect_label, cohens_f in effect_sizes.items():

    # Convert Cohen's f to eta-squared for Pingouin.
    f_squared = cohens_f ** 2
    eta_squared = f_squared / (1 + f_squared)

    # Required complete sample for 80% power.
    calculated_n = pg.power_rm_anova(
        eta_squared=eta_squared,
        m=N_ALGORITHMS,
        n=None,
        power=TARGET_POWER,
        alpha=ALPHA,
        corr=AVERAGE_CORRELATION,
        epsilon=EPSILON,
    )

    required_n = math.ceil(calculated_n)

    # Estimated power of the current pilot sample.
    pilot_power = pg.power_rm_anova(
        eta_squared=eta_squared,
        m=N_ALGORITHMS,
        n=PILOT_N,
        power=None,
        alpha=ALPHA,
        corr=AVERAGE_CORRELATION,
        epsilon=EPSILON,
    )

    power_rows.append(
        {
            "Assumed effect": effect_label,
            "Cohen's f": cohens_f,
            "Required N for 80% power": required_n,
            "Estimated power with N=9": pilot_power,
        }
    )


power_analysis_df = pd.DataFrame(power_rows)

display(
    power_analysis_df.style.format(
        {
            "Cohen's f": "{:.2f}",
            "Estimated power with N=9": "{:.1%}",
        }
    )
)

,Assumed effect,Cohen's f,Required N for 80% power,Estimated power with N=9
0,Small,0.10,273,2.1%
1,Medium,0.25,46,9.8%
2,Large,0.40,19,33.4%


# Evaluation – Pilot Study (n=9)

In [4]:
json_files = sorted(DATA_DIR.glob("*.json"))

if not json_files:
    raise FileNotFoundError(
        f"No JSON files were found in {DATA_DIR}"
    )


evaluations = []

for path in json_files:
    with path.open("r", encoding="utf-8") as file:
        data = json.load(file)

    if not isinstance(data, dict):
        raise ValueError(
            f"{path.name} does not contain a top-level JSON object."
        )

    evaluations.append(
        {
            "source_file": path.name,
            "created_at": pd.to_datetime(
                data.get("created_at"),
                utc=True,
                errors="coerce",
            ),
            "data": data,
        }
    )


# Sort chronologically. Files without a valid timestamp come last.
evaluations.sort(
    key=lambda item: (
        pd.isna(item["created_at"]),
        (
            item["created_at"]
            if pd.notna(item["created_at"])
            else pd.Timestamp("2262-01-01", tz="UTC")
        ),
        item["source_file"],
    )
)


# Assign anonymous evaluator IDs internally.
for evaluator_id, evaluation in enumerate(
    evaluations,
    start=1,
):
    evaluation["evaluator_id"] = evaluator_id


print(f"Loaded {len(evaluations)} JSON files.")
print("Input folder:", DATA_DIR)
print("Output folder:", OUTPUT_DIR)

if evaluations:
    print("\nFirst file:", evaluations[0]["source_file"])
    print(
        "Top-level fields:",
        list(evaluations[0]["data"].keys()),
    )

Loaded 9 JSON files.
Input folder: /app/evaluation_results
Output folder: /app/evaluation_results/evaluation_outputs

First file: evaluation_20260622_113107_28bbe2bdbbdb49d980abf633de64f31d.json
Top-level fields: ['created_at', 'reference_movie_id', 'responses', 'overall']


## Data Aggregation

### Evaluator x Movie x Algorithm
one row per evaluator × movie × algorithm

In [5]:
evaluator_movie_algorithm_rows = []

for evaluation in evaluations:
    evaluator_id = evaluation["evaluator_id"]
    source_file = evaluation["source_file"]
    data = evaluation["data"]

    overall = data.get("overall", {}) or {}
    responses = data.get("responses", []) or []

    top_5 = {
        str(movie_id)
        for movie_id in (
            overall.get("top_5", []) or []
        )
    }

    for response_number, response in enumerate(
        responses,
        start=1,
    ):
        movie_id_raw = response.get("movie_id")

        if movie_id_raw is None:
            raise ValueError(
                f"{source_file}, response {response_number}: "
                "movie_id is missing."
            )

        movie_id = str(movie_id_raw)

        rating = response.get("rating")
        watch_likelihood = response.get(
            "watch_likelihood"
        )

        # Exactly one field must provide the intention value.
        if rating is None and watch_likelihood is None:
            raise ValueError(
                f"{source_file}, movie {movie_id}: "
                "rating and watch_likelihood are both missing."
            )

        if rating is not None and watch_likelihood is not None:
            raise ValueError(
                f"{source_file}, movie {movie_id}: "
                "rating and watch_likelihood are both populated."
            )

        if watch_likelihood is not None:
            viewing_intention_score = watch_likelihood
            intention_source = "watch_likelihood"
        else:
            viewing_intention_score = rating
            intention_source = "rating"

        algorithms = response.get(
            "algorithms",
            {},
        ) or {}

        if not algorithms:
            raise ValueError(
                f"{source_file}, movie {movie_id}: "
                "no algorithm attribution was found."
            )

        for algorithm, algorithm_rank in algorithms.items():
            evaluator_movie_algorithm_rows.append(
                {
                    "evaluator_id": evaluator_id,
                    # "source_file": source_file,
                    "reference_movie_id": data.get(
                        "reference_movie_id"
                    ),
                    "response_number": response_number,
                    "movie_id": movie_id,
                    "algorithm": algorithm,
                    "algorithm_rank": algorithm_rank,
                    "familiarity": response.get(
                        "familiarity"
                    ),
                    "rating": rating,
                    "watch_likelihood": watch_likelihood,
                    "viewing_intention_score": viewing_intention_score,
                    "intention_source": intention_source,
                    "relevance": response.get(
                        "relevance"
                    ),
                    "novelty_score": response.get(
                        "novelty_score"
                    ),
                    "in_top5": int(
                        movie_id in top_5
                    ),
                    "n_algorithms_for_movie": len(
                        algorithms
                    ),
                    "is_shared_recommendation": int(
                        len(algorithms) > 1
                    ),
                }
            )


evaluator_movie_algorithm_df = pd.DataFrame(
    evaluator_movie_algorithm_rows
)


# Convert analytical columns to numeric values.
numeric_columns = [
    "evaluator_id",
    "response_number",
    "algorithm_rank",
    "rating",
    "watch_likelihood",
    "viewing_intention_score",
    "relevance",
    "novelty_score",
    "in_top5",
    "n_algorithms_for_movie",
    "is_shared_recommendation",
]

for column in numeric_columns:
    evaluator_movie_algorithm_df[column] = pd.to_numeric(
        evaluator_movie_algorithm_df[column],
        errors="coerce",
    )


# Fixed-scale normalized values.
evaluator_movie_algorithm_df["relevance_normalized"] = (
    evaluator_movie_algorithm_df["relevance"] - 1
) / 4

evaluator_movie_algorithm_df["intention_normalized"] = (
    evaluator_movie_algorithm_df["viewing_intention_score"] - 1
) / 4

evaluator_movie_algorithm_df["novelty_normalized"] = (
    evaluator_movie_algorithm_df["novelty_score"] / 2
)


# Exploratory serendipity score.
evaluator_movie_algorithm_df["serendipity_score"] = (
    evaluator_movie_algorithm_df["novelty_normalized"]
    * (
        evaluator_movie_algorithm_df["relevance_normalized"]
        + evaluator_movie_algorithm_df["intention_normalized"]
    )
    / 2
)


# Sort by the natural row hierarchy.
evaluator_movie_algorithm_df = (
    evaluator_movie_algorithm_df
    .sort_values(
        [
            "evaluator_id",
            "algorithm",
            "algorithm_rank",
            "movie_id",
        ]
    )
    .reset_index(drop=True)
)


output_path = (
    OUTPUT_DIR
    / "results_evaluator_movie_algorithm.csv"
)

evaluator_movie_algorithm_df.to_csv(
    output_path,
    index=False,
)


print("Shape:", evaluator_movie_algorithm_df.shape)
print("Saved:", output_path)

display(
    evaluator_movie_algorithm_df.head(10)
)

Shape: (360, 20)
Saved: /app/evaluation_results/evaluation_outputs/results_evaluator_movie_algorithm.csv


,evaluator_id,reference_movie_id,response_number,movie_id,algorithm,algorithm_rank,familiarity,rating,watch_likelihood,viewing_intention_score,intention_source,relevance,novelty_score,in_top5,n_algorithms_for_movie,is_shared_recommendation,relevance_normalized,intention_normalized,novelty_normalized,serendipity_score
0,1,2628,1,79006,castoverlap,1,unknown,NaN,3.0,3,watch_likelihood,4,2,0,2,1,0.75,0.50,1.0,0.6250
1,1,2628,17,82854,castoverlap,2,heard,NaN,3.0,3,watch_likelihood,2,1,0,1,0,0.25,0.50,0.5,0.1875
2,1,2628,29,101025,castoverlap,3,unknown,NaN,4.0,4,watch_likelihood,2,2,0,1,0,0.25,0.75,1.0,0.5000
3,1,2628,13,2115,castoverlap,4,heard,NaN,2.0,2,watch_likelihood,2,1,0,2,1,0.25,0.25,0.5,0.1250
4,1,2628,10,1291,castoverlap,5,heard,NaN,2.0,2,watch_likelihood,2,1,0,1,0,0.25,0.25,0.5,0.1250
5,1,2628,18,61160,genome_overlap,1,unknown,NaN,3.0,3,watch_likelihood,5,2,1,5,1,1.00,0.50,1.0,0.7500
6,1,2628,22,68358,genome_overlap,2,heard,NaN,3.0,3,watch_likelihood,4,1,0,1,0,0.75,0.50,0.5,0.3125
7,1,2628,3,1356,genome_overlap,3,heard,NaN,2.0,2,watch_likelihood,4,1,0,1,0,0.75,0.25,0.5,0.2500
8,1,2628,21,5944,genome_overlap,4,heard,NaN,3.0,3,watch_likelihood,4,1,0,1,0,0.75,0.50,0.5,0.3125
9,1,2628,19,8371,genome_overlap,5,unknown,NaN,2.0,2,watch_likelihood,3,2,0,1,0,0.50,0.25,1.0,0.3750


### evaluator × algorithm

In [6]:
# Confirm that every evaluator × algorithm cell contains five recommendations.
cell_sizes = (
    evaluator_movie_algorithm_df
    .groupby(
        ["evaluator_id", "algorithm"],
        observed=True,
    )
    .size()
)

invalid_cells = cell_sizes[
    cell_sizes != 5
]

if not invalid_cells.empty:
    raise ValueError(
        "Some evaluator × algorithm cells do not contain "
        f"exactly five recommendations:\n{invalid_cells}"
    )


# Aggregate from evaluator × movie × algorithm
# to evaluator × algorithm.
evaluator_algorithm_df = (
    evaluator_movie_algorithm_df
    .groupby(
        ["evaluator_id", "algorithm"],
        observed=True,
    )
    .agg(
        n_recommendations=(
            "movie_id",
            "size",
        ),

        mean_relevance=(
            "relevance",
            "mean",
        ),
        median_relevance=(
            "relevance",
            "median",
        ),

        mean_viewing_intention_score=(
            "viewing_intention_score",
            "mean",
        ),
        median_viewing_intention_score=(
            "viewing_intention_score",
            "median",
        ),

        mean_novelty_score=(
            "novelty_score",
            "mean",
        ),
        median_novelty_score=(
            "novelty_score",
            "median",
        ),

        top5_hit_count=(
            "in_top5",
            "sum",
        ),

        rating_count=(
            "intention_source",
            lambda values: int(
                (values == "rating").sum()
            ),
        ),

        watch_likelihood_count=(
            "intention_source",
            lambda values: int(
                (values == "watch_likelihood").sum()
            ),
        ),

        mean_serendipity_score=(
            "serendipity_score",
            "mean",
        ),

        shared_recommendation_count=(
            "is_shared_recommendation",
            "sum",
        ),
    )
    .reset_index()
)


# Top-5 measures.
evaluator_algorithm_df["top5_hit_rate"] = (
    evaluator_algorithm_df["top5_hit_count"]
    / evaluator_algorithm_df["n_recommendations"]
)

evaluator_algorithm_df["any_top5_hit"] = (
    evaluator_algorithm_df["top5_hit_count"] > 0
).astype(int)


# intention-source proportions.
evaluator_algorithm_df["rating_share"] = (
    evaluator_algorithm_df["rating_count"]
    / evaluator_algorithm_df["n_recommendations"]
)

evaluator_algorithm_df["watch_likelihood_share"] = (
    evaluator_algorithm_df["watch_likelihood_count"]
    / evaluator_algorithm_df["n_recommendations"]
)


# Sort consistently.
evaluator_algorithm_df = (
    evaluator_algorithm_df
    .sort_values(
        ["evaluator_id", "algorithm"]
    )
    .reset_index(drop=True)
)


# Confirm the complete repeated-measures matrix.
n_evaluators = (
    evaluator_algorithm_df["evaluator_id"]
    .nunique()
)

n_algorithms = (
    evaluator_algorithm_df["algorithm"]
    .nunique()
)

expected_rows = n_evaluators * n_algorithms

if len(evaluator_algorithm_df) != expected_rows:
    raise ValueError(
        "The evaluator × algorithm matrix is incomplete. "
        f"Expected {expected_rows} rows, "
        f"found {len(evaluator_algorithm_df)}."
    )


# Save the main statistical dataset.
output_path = (
    OUTPUT_DIR
    / "results_evaluator_algorithm.csv"
)

evaluator_algorithm_df.to_csv(
    output_path,
    index=False,
)


print("Shape:", evaluator_algorithm_df.shape)
print("Evaluators:", n_evaluators)
print("Algorithms:", n_algorithms)
print("Saved:", output_path)

display(
    evaluator_algorithm_df.head(16)
)

Shape: (72, 18)
Evaluators: 9
Algorithms: 8
Saved: /app/evaluation_results/evaluation_outputs/results_evaluator_algorithm.csv


,evaluator_id,algorithm,n_recommendations,mean_relevance,median_relevance,mean_viewing_intention_score,median_viewing_intention_score,mean_novelty_score,median_novelty_score,top5_hit_count,rating_count,watch_likelihood_count,mean_serendipity_score,shared_recommendation_count,top5_hit_rate,any_top5_hit,rating_share,watch_likelihood_share
0,1,castoverlap,5,2.4,2.0,2.8,3.0,1.4,1.0,0,0,5,0.3125,2,0.0,0,0.0,1.0
1,1,genome_overlap,5,4.0,4.0,2.6,3.0,1.4,1.0,1,0,5,0.4000,1,0.2,1,0.0,1.0
2,1,image_similarity,5,3.2,3.0,3.0,3.0,1.4,2.0,2,1,4,0.3875,2,0.4,1,0.2,0.8
3,1,image_text_similarity,5,2.8,2.0,3.0,3.0,1.0,1.0,3,2,3,0.2125,2,0.6,1,0.4,0.6
4,1,knn,5,2.8,3.0,3.4,3.0,0.4,0.0,1,3,2,0.1000,1,0.2,1,0.6,0.4
5,1,subtitles,5,3.6,4.0,3.0,3.0,2.0,2.0,1,0,5,0.5750,1,0.2,1,0.0,1.0
6,1,tmdb,5,3.6,4.0,3.2,3.0,1.2,1.0,1,1,4,0.3250,0,0.2,1,0.2,0.8
7,1,weighted_hybrid,5,2.8,2.0,2.8,3.0,1.2,1.0,2,1,4,0.2625,3,0.4,1,0.2,0.8
8,2,castoverlap,5,2.4,2.0,1.8,2.0,2.0,2.0,0,0,5,0.2750,0,0.0,0,0.0,1.0
9,2,genome_overlap,5,3.2,3.0,2.6,3.0,1.8,2.0,0,0,5,0.4125,2,0.0,0,0.0,1.0


### Overall Results (aggregated)

In [7]:
overall_rows = []

for evaluation in evaluations:
    data = evaluation["data"]
    overall = data.get("overall", {}) or {}

    overall_rows.append(
        {
            "evaluator_id": evaluation["evaluator_id"],
            "overall_relevance": overall.get("relevance"),
            "overall_diversity": overall.get("diversity"),
            "overall_satisfaction": overall.get("satisfaction"),
            "overall_trust": overall.get("trust"),
            "overall_usefulness": overall.get("usefulness"),
        }
    )


overall_ratings_df = (
    pd.DataFrame(overall_rows)
    .sort_values("evaluator_id")
    .reset_index(drop=True)
)


numeric_columns = [
    "evaluator_id",
    "overall_relevance",
    "overall_diversity",
    "overall_satisfaction",
    "overall_trust",
    "overall_usefulness",
]

for column in numeric_columns:
    overall_ratings_df[column] = pd.to_numeric(
        overall_ratings_df[column],
        errors="coerce",
    )


output_path = (
    OUTPUT_DIR
    / "results_overall_ratings.csv"
)

overall_ratings_df.to_csv(
    output_path,
    index=False,
)


print("Shape:", overall_ratings_df.shape)
print("Saved:", output_path)

display(overall_ratings_df)

Shape: (9, 6)
Saved: /app/evaluation_results/evaluation_outputs/results_overall_ratings.csv


,evaluator_id,overall_relevance,overall_diversity,overall_satisfaction,overall_trust,overall_usefulness
0,1,3,5,4,2,3
1,2,4,3,4,3,4
2,3,4,4,5,5,4
3,4,2,5,2,1,1
4,5,4,3,4,4,3
5,6,3,5,4,5,5
6,7,3,5,3,5,3
7,8,4,2,5,5,4
8,9,4,5,3,3,3


## Descriptive Statistics

In [8]:
algorithm_descriptives_df = (
    evaluator_algorithm_df
    .groupby("algorithm", observed=True)
    .agg(
        n_evaluators=("evaluator_id", "nunique"),

        mean_relevance=("mean_relevance", "mean"),
        median_relevance=("mean_relevance", "median"),
        sd_relevance=("mean_relevance", "std"),

        mean_intention=("mean_viewing_intention_score", "mean"),
        median_intention=("mean_viewing_intention_score", "median"),
        sd_intention=("mean_viewing_intention_score", "std"),

        mean_novelty=("mean_novelty_score", "mean"),
        median_novelty=("mean_novelty_score", "median"),
        sd_novelty=("mean_novelty_score", "std"),

        mean_top5_hit_count=("top5_hit_count", "mean"),
        sd_top5_hit_count=("top5_hit_count", "std"),

        mean_top5_hit_rate=("top5_hit_rate", "mean"),
        sd_top5_hit_rate=("top5_hit_rate", "std"),

        evaluators_with_any_top5_hit=("any_top5_hit", "sum"),
    )
    .reset_index()
)

display(algorithm_descriptives_df.round(3))

,algorithm,n_evaluators,mean_relevance,median_relevance,sd_relevance,mean_intention,median_intention,sd_intention,mean_novelty,median_novelty,sd_novelty,mean_top5_hit_count,sd_top5_hit_count,mean_top5_hit_rate,sd_top5_hit_rate,evaluators_with_any_top5_hit
0,castoverlap,9,3.022,3.2,0.809,2.978,2.8,0.751,1.422,1.4,0.514,0.778,0.833,0.156,0.167,5
1,genome_overlap,9,3.467,3.6,0.800,3.289,3.2,0.775,1.267,1.4,0.458,0.889,0.782,0.178,0.156,6
2,image_similarity,9,3.400,3.4,0.883,3.311,3.2,0.694,1.222,1.2,0.595,1.222,0.833,0.244,0.167,7
3,image_text_similarity,9,3.467,3.4,0.917,3.444,3.0,0.835,1.089,1.2,0.679,1.556,1.130,0.311,0.226,7
4,knn,9,3.333,3.4,0.566,3.889,4.0,0.481,0.844,0.8,0.384,1.333,0.500,0.267,0.100,9
5,subtitles,9,2.844,2.4,0.792,2.422,2.4,0.869,1.689,2.0,0.437,0.333,0.500,0.067,0.100,3
6,tmdb,9,2.978,3.0,0.886,3.022,3.0,0.784,1.422,1.6,0.552,0.556,0.726,0.111,0.145,4
7,weighted_hybrid,9,3.667,3.6,0.843,3.644,3.6,0.786,1.044,1.2,0.590,1.333,0.866,0.267,0.173,7


In [9]:
# Create a compact, publication-ready table.
algorithm_latex_df = (
    algorithm_descriptives_df
    .copy()
    .sort_values(
        "mean_intention",
        ascending=False,
    )
)


# Optional: use more readable algorithm names.
algorithm_name_map = {
    "castoverlap": "Cast overlap",
    "genome_overlap": "Genome overlap",
    "image_similarity": "Image similarity",
    "image_text_similarity": "Image--text similarity",
    "knn": "KNN",
    "subtitles": "Subtitles",
    "tmdb": "TMDB",
    "weighted_hybrid": "Weighted hybrid",
}

algorithm_latex_df["Algorithm"] = (
    algorithm_latex_df["algorithm"]
    .map(algorithm_name_map)
    .fillna(
        algorithm_latex_df["algorithm"]
        .str.replace("_", " ", regex=False)
        .str.title()
    )
)


# Combine means and standard deviations into single columns.
algorithm_latex_df[
    r"Relevance $M$ ($SD$)"
] = algorithm_latex_df.apply(
    lambda row: (
        f"{row['mean_relevance']:.2f} "
        f"({row['sd_relevance']:.2f})"
    ),
    axis=1,
)

algorithm_latex_df[
    r"Intention $M$ ($SD$)"
] = algorithm_latex_df.apply(
    lambda row: (
        f"{row['mean_intention']:.2f} "
        f"({row['sd_intention']:.2f})"
    ),
    axis=1,
)

algorithm_latex_df[
    r"Novelty $M$ ($SD$)"
] = algorithm_latex_df.apply(
    lambda row: (
        f"{row['mean_novelty']:.2f} "
        f"({row['sd_novelty']:.2f})"
    ),
    axis=1,
)


# Format the mean Top-5 hit rate as a percentage.
algorithm_latex_df[
    r"Top-5 hit rate $M$"
] = algorithm_latex_df[
    "mean_top5_hit_rate"
].map(
    lambda value: f"{value * 100:.1f}\\%"
)

# Retain only the columns intended for the report.
algorithm_latex_df = algorithm_latex_df[
    [
        "Algorithm",
        r"Relevance $M$ ($SD$)",
        r"Intention $M$ ($SD$)",
        r"Novelty $M$ ($SD$)",
        r"Top-5 hit rate $M$",
    ]
]

display(algorithm_latex_df)

latex_code = algorithm_latex_df.to_latex(
    index=False,
    escape=False,
    caption=(
        "Descriptive statistics for the eight recommendation "
        "algorithms across nine evaluators."
    ),
    label="tab:algorithm-descriptives",
    position="htbp",
    column_format="lcccc",
)

latex_code = latex_code.replace(
    "\\begin{table}[htbp]\n",
    "\\begin{table}[htbp]\n\\centering\n",
)

latex_path = (
    OUTPUT_DIR
    / "table_algorithm_descriptives.tex"
)

latex_path.write_text(
    latex_code,
    encoding="utf-8",
)

print("Saved:", latex_path)
print()
print(latex_code)

,Algorithm,Relevance $M$ ($SD$),Intention $M$ ($SD$),Novelty $M$ ($SD$),Top-5 hit rate $M$
4,KNN,3.33 (0.57),3.89 (0.48),0.84 (0.38),26.7\%
7,Weighted hybrid,3.67 (0.84),3.64 (0.79),1.04 (0.59),26.7\%
3,Image--text similarity,3.47 (0.92),3.44 (0.84),1.09 (0.68),31.1\%
2,Image similarity,3.40 (0.88),3.31 (0.69),1.22 (0.60),24.4\%
1,Genome overlap,3.47 (0.80),3.29 (0.78),1.27 (0.46),17.8\%
6,TMDB,2.98 (0.89),3.02 (0.78),1.42 (0.55),11.1\%
0,Cast overlap,3.02 (0.81),2.98 (0.75),1.42 (0.51),15.6\%
5,Subtitles,2.84 (0.79),2.42 (0.87),1.69 (0.44),6.7\%


Saved: /app/evaluation_results/evaluation_outputs/table_algorithm_descriptives.tex

\begin{table}[htbp]
\centering
\caption{Descriptive statistics for the eight recommendation algorithms across nine evaluators.}
\label{tab:algorithm-descriptives}
\begin{tabular}{lcccc}
\toprule
Algorithm & Relevance $M$ ($SD$) & Intention $M$ ($SD$) & Novelty $M$ ($SD$) & Top-5 hit rate $M$ \\
\midrule
KNN & 3.33 (0.57) & 3.89 (0.48) & 0.84 (0.38) & 26.7\% \\
Weighted hybrid & 3.67 (0.84) & 3.64 (0.79) & 1.04 (0.59) & 26.7\% \\
Image--text similarity & 3.47 (0.92) & 3.44 (0.84) & 1.09 (0.68) & 31.1\% \\
Image similarity & 3.40 (0.88) & 3.31 (0.69) & 1.22 (0.60) & 24.4\% \\
Genome overlap & 3.47 (0.80) & 3.29 (0.78) & 1.27 (0.46) & 17.8\% \\
TMDB & 2.98 (0.89) & 3.02 (0.78) & 1.42 (0.55) & 11.1\% \\
Cast overlap & 3.02 (0.81) & 2.98 (0.75) & 1.42 (0.51) & 15.6\% \\
Subtitles & 2.84 (0.79) & 2.42 (0.87) & 1.69 (0.44) & 6.7\% \\
\bottomrule
\end{tabular}
\end{table}



## Inferential Statistics

In [10]:
from itertools import combinations

import numpy as np
from scipy.stats import friedmanchisquare, rankdata, wilcoxon
from statsmodels.stats.multitest import multipletests


### Omnibus Tests

The same evaluators assessed all eight algorithms, so the algorithm outcomes are
compared with Friedman repeated-measures tests. The four outcomes (relevance,
viewing intention, novelty, and Top-5 hit rate) are treated as a family of
planned primary endpoints, so Holm correction (family-wise alpha = .05) is
applied across the four omnibus tests. Kendall's $W$ is reported as the
omnibus effect size.


In [11]:
# =============================================================================
# FRIEDMAN OMNIBUS TESTS AND REPORTING
# =============================================================================

import numpy as np
import pandas as pd

from scipy.stats import friedmanchisquare
from statsmodels.stats.multitest import multipletests


# Family-wise alpha across the four planned omnibus tests.
# Holm correction below controls this rate; do not pre-divide it.
ALPHA = 0.05

outcomes = {
    "Relevance": "mean_relevance",
    "Viewing intention": "mean_viewing_intention_score",
    "Novelty": "mean_novelty_score",
    "Top-5 hit count": "top5_hit_count",
}

algorithm_order = sorted(
    evaluator_algorithm_df["algorithm"].unique()
)

omnibus_rows = []


# -----------------------------------------------------------------------------
# Run one Friedman test for each outcome
# -----------------------------------------------------------------------------

for outcome_name, value_column in outcomes.items():

    outcome_wide = (
        evaluator_algorithm_df
        .pivot(
            index="evaluator_id",
            columns="algorithm",
            values=value_column,
        )
        .reindex(columns=algorithm_order)
        .sort_index()
    )

    if outcome_wide.isna().any().any():
        raise ValueError(
            f"Missing values found for {outcome_name}."
        )

    test_result = friedmanchisquare(
        *[
            outcome_wide[algorithm].to_numpy()
            for algorithm in algorithm_order
        ]
    )

    n_evaluators = outcome_wide.shape[0]
    n_algorithms = outcome_wide.shape[1]

    # Kendall's W omnibus effect size.
    kendalls_w = (
        test_result.statistic
        / (
            n_evaluators
            * (n_algorithms - 1)
        )
    )

    omnibus_rows.append(
        {
            "outcome": outcome_name,
            "statistic": float(test_result.statistic),
            "df": n_algorithms - 1,
            "p_value": float(test_result.pvalue),
            "kendalls_w": float(kendalls_w),
        }
    )


friedman_results_df = pd.DataFrame(
    omnibus_rows
)


# -----------------------------------------------------------------------------
# Apply Holm correction across the four omnibus tests
# (family-wise alpha = 0.05), matching the planning assumption
# used in the power analysis and the post-hoc correction below.
# -----------------------------------------------------------------------------

holm_rejected, holm_p_values, _, _ = multipletests(
    friedman_results_df["p_value"].to_numpy(),
    alpha=ALPHA,
    method="holm",
)

friedman_results_df["p_holm"] = holm_p_values
friedman_results_df["significant"] = holm_rejected

friedman_results_df["result"] = np.where(
    friedman_results_df["significant"],
    "Significant",
    "Not significant",
)


# -----------------------------------------------------------------------------
# Display numerical results
# -----------------------------------------------------------------------------

display(
    friedman_results_df.round(
        {
            "statistic": 3,
            "p_value": 4,
            "kendalls_w": 3,
        }
    )
)


# -----------------------------------------------------------------------------
# Save complete numerical results
# -----------------------------------------------------------------------------

csv_path = (
    OUTPUT_DIR
    / "results_friedman_omnibus.csv"
)

friedman_results_df.to_csv(
    csv_path,
    index=False,
)


# -----------------------------------------------------------------------------
# Create report-ready LaTeX table
# -----------------------------------------------------------------------------

def format_p_value(value):
    if value < 0.001:
        return r"$< .001$"

    return f"{value:.3f}".replace(
        "0.",
        ".",
    )


friedman_latex_df = pd.DataFrame(
    {
        "Outcome": friedman_results_df["outcome"],
        r"$\chi^2$": (
            friedman_results_df["statistic"]
            .map(lambda value: f"{value:.2f}")
        ),
        r"$df$": friedman_results_df["df"].astype(int),
        r"$p$": (
            friedman_results_df["p_value"]
            .map(format_p_value)
        ),
        r"$p_{\mathrm{Holm}}$": (
            friedman_results_df["p_holm"]
            .map(format_p_value)
        ),
        r"Kendall's $W$": (
            friedman_results_df["kendalls_w"]
            .map(lambda value: f"{value:.3f}")
        ),
        "Result": friedman_results_df["result"],
    }
)

display(friedman_latex_df)


latex_code = friedman_latex_df.to_latex(
    index=False,
    escape=False,
    caption=(
        "Friedman omnibus tests comparing the eight "
        "recommendation algorithms across nine evaluators."
    ),
    label="tab:friedman-omnibus",
    position="htbp",
    column_format="lcccccc",
)

latex_code = latex_code.replace(
    "\\begin{table}[htbp]\n",
    "\\begin{table}[htbp]\n\\centering\n",
)

latex_path = (
    OUTPUT_DIR
    / "table_friedman_omnibus.tex"
)

latex_path.write_text(
    latex_code,
    encoding="utf-8",
)

print("Saved:", csv_path)
print("Saved:", latex_path)
print()
print(latex_code)

,outcome,statistic,df,p_value,kendalls_w,p_holm,significant,result
0,Relevance,14.020,7,0.0508,0.223,0.078917,False,Not significant
1,Viewing intention,23.672,7,0.0013,0.376,0.005203,True,Significant
2,Novelty,19.227,7,0.0075,0.305,0.022514,True,Significant
3,Top-5 hit count,14.742,7,0.0395,0.234,0.078917,False,Not significant


,Outcome,$\chi^2$,$df$,$p$,$p_{\mathrm{Holm}}$,Kendall's $W$,Result
0,Relevance,14.02,7,.051,.079,0.223,Not significant
1,Viewing intention,23.67,7,.001,.005,0.376,Significant
2,Novelty,19.23,7,.008,.023,0.305,Significant
3,Top-5 hit count,14.74,7,.039,.079,0.234,Not significant


Saved: /app/evaluation_results/evaluation_outputs/results_friedman_omnibus.csv
Saved: /app/evaluation_results/evaluation_outputs/table_friedman_omnibus.tex

\begin{table}[htbp]
\centering
\caption{Friedman omnibus tests comparing the eight recommendation algorithms across nine evaluators.}
\label{tab:friedman-omnibus}
\begin{tabular}{lcccccc}
\toprule
Outcome & $\chi^2$ & $df$ & $p$ & $p_{\mathrm{Holm}}$ & Kendall's $W$ & Result \\
\midrule
Relevance & 14.02 & 7 & .051 & .079 & 0.223 & Not significant \\
Viewing intention & 23.67 & 7 & .001 & .005 & 0.376 & Significant \\
Novelty & 19.23 & 7 & .008 & .023 & 0.305 & Significant \\
Top-5 hit count & 14.74 & 7 & .039 & .079 & 0.234 & Not significant \\
\bottomrule
\end{tabular}
\end{table}



### Post-hoc Pairwise Comparisons

Pairwise Wilcoxon signed-rank tests are run only for outcomes whose
Holm-adjusted Friedman test is significant. Holm correction is then applied
across the 28 algorithm pairs within each eligible outcome.


In [12]:
# =============================================================================
# POST-HOC PAIRWISE COMPARISONS
# Pairwise Wilcoxon signed-rank tests with Holm correction per outcome
# =============================================================================

from itertools import combinations

import numpy as np
import pandas as pd

from scipy.stats import rankdata, wilcoxon
from statsmodels.stats.multitest import multipletests


ALPHA = 0.05

# These outcome names must match friedman_results_df exactly.
POSTHOC_OUTCOMES = {
    "Relevance": "mean_relevance",
    "Viewing intention": "mean_viewing_intention_score",
    "Novelty": "mean_novelty_score",
    "Top-5 hit count": "top5_hit_count",
}

ALGORITHM_NAME_MAP = {
    "castoverlap": "Cast overlap",
    "genome_overlap": "Genome overlap",
    "image_similarity": "Image similarity",
    "image_text_similarity": "Image–text similarity",
    "knn": "KNN",
    "subtitles": "Subtitles",
    "tmdb": "TMDB",
    "weighted_hybrid": "Weighted hybrid",
}


def matched_pairs_rank_biserial(values_a, values_b):
    """
    Calculate the matched-pairs rank-biserial correlation.

    Positive values indicate that algorithm A tends to score higher.
    Negative values indicate that algorithm B tends to score higher.
    """
    differences = (
        np.asarray(values_a, dtype=float)
        - np.asarray(values_b, dtype=float)
    )

    differences = np.round(
        differences,
        decimals=10,
    )

    # Match Wilcoxon's zero_method="wilcox":
    # remove pairs with a zero difference.
    nonzero_differences = differences[
        differences != 0
    ]

    if len(nonzero_differences) == 0:
        return 0.0

    ranks = rankdata(
        np.abs(nonzero_differences),
        method="average",
    )

    positive_rank_sum = ranks[
        nonzero_differences > 0
    ].sum()

    negative_rank_sum = ranks[
        nonzero_differences < 0
    ].sum()

    total_rank_sum = (
        positive_rank_sum
        + negative_rank_sum
    )

    return float(
        (
            positive_rank_sum
            - negative_rank_sum
        )
        / total_rank_sum
    )


# -----------------------------------------------------------------------------
# Validate the Friedman results
# -----------------------------------------------------------------------------

required_friedman_columns = {
    "outcome",
    "significant",
}

missing_friedman_columns = (
    required_friedman_columns
    - set(friedman_results_df.columns)
)

if missing_friedman_columns:
    raise ValueError(
        "friedman_results_df is missing these columns: "
        f"{sorted(missing_friedman_columns)}"
    )


# Select outcomes using the current unadjusted Friedman results.
significant_outcomes = (
    friedman_results_df.loc[
        friedman_results_df["significant"],
        "outcome",
    ]
    .tolist()
)

unknown_outcomes = (
    set(significant_outcomes)
    - set(POSTHOC_OUTCOMES)
)

if unknown_outcomes:
    raise ValueError(
        "These significant outcomes are missing from "
        f"POSTHOC_OUTCOMES: {sorted(unknown_outcomes)}"
    )


print(
    "Outcomes selected for post-hoc testing:",
    significant_outcomes
    if significant_outcomes
    else "None",
)


# Important: recreate the results from scratch.
all_posthoc_results = []


# -----------------------------------------------------------------------------
# Run pairwise Wilcoxon tests
# -----------------------------------------------------------------------------

for outcome_name in significant_outcomes:

    value_column = POSTHOC_OUTCOMES[
        outcome_name
    ]

    if value_column not in evaluator_algorithm_df.columns:
        raise ValueError(
            f"Column '{value_column}' is missing from "
            "evaluator_algorithm_df."
        )

    outcome_wide = (
        evaluator_algorithm_df
        .pivot(
            index="evaluator_id",
            columns="algorithm",
            values=value_column,
        )
        .sort_index()
        .sort_index(axis=1)
    )

    if outcome_wide.isna().any().any():
        missing_rows = (
            outcome_wide
            .isna()
            .any(axis=1)
        )

        raise ValueError(
            f"{outcome_name} contains missing values for "
            f"{int(missing_rows.sum())} evaluator(s)."
        )

    outcome_rows = []

    for algorithm_a, algorithm_b in combinations(
        outcome_wide.columns,
        2,
    ):

        values_a = np.round(
            outcome_wide[
                algorithm_a
            ].to_numpy(dtype=float),
            decimals=10,
        )

        values_b = np.round(
            outcome_wide[
                algorithm_b
            ].to_numpy(dtype=float),
            decimals=10,
        )

        differences = np.round(
            values_a - values_b,
            decimals=10,
        )

        n_pairs = len(differences)
        n_a_higher = int(
            (differences > 0).sum()
        )
        n_b_higher = int(
            (differences < 0).sum()
        )
        n_equal = int(
            (differences == 0).sum()
        )
        n_nonzero = (
            n_pairs - n_equal
        )

        # Wilcoxon is undefined when all paired differences are zero.
        if n_nonzero == 0:
            wilcoxon_statistic = 0.0
            p_uncorrected = 1.0
            rank_biserial_r = 0.0

        else:
            test_result = wilcoxon(
                values_a,
                values_b,
                alternative="two-sided",
                zero_method="wilcox",
                correction=False,
                method="auto",
            )

            wilcoxon_statistic = float(
                test_result.statistic
            )

            p_uncorrected = float(
                test_result.pvalue
            )

            rank_biserial_r = (
                matched_pairs_rank_biserial(
                    values_a,
                    values_b,
                )
            )

        if rank_biserial_r > 0:
            direction = (
                f"{ALGORITHM_NAME_MAP.get(algorithm_a, algorithm_a)} "
                "higher"
            )

        elif rank_biserial_r < 0:
            direction = (
                f"{ALGORITHM_NAME_MAP.get(algorithm_b, algorithm_b)} "
                "higher"
            )

        else:
            direction = "No directional difference"

        outcome_rows.append(
            {
                "outcome": outcome_name,
                "algorithm_a": algorithm_a,
                "algorithm_b": algorithm_b,
                "algorithm_a_label": (
                    ALGORITHM_NAME_MAP.get(
                        algorithm_a,
                        algorithm_a,
                    )
                ),
                "algorithm_b_label": (
                    ALGORITHM_NAME_MAP.get(
                        algorithm_b,
                        algorithm_b,
                    )
                ),
                "mean_a": float(
                    np.mean(values_a)
                ),
                "mean_b": float(
                    np.mean(values_b)
                ),
                "median_a": float(
                    np.median(values_a)
                ),
                "median_b": float(
                    np.median(values_b)
                ),
                "mean_difference_a_minus_b": float(
                    np.mean(differences)
                ),
                "median_difference_a_minus_b": float(
                    np.median(differences)
                ),
                "wilcoxon_w": wilcoxon_statistic,
                "p_uncorrected": p_uncorrected,
                "rank_biserial_r": rank_biserial_r,
                "direction": direction,
                "n_pairs": n_pairs,
                "n_nonzero": n_nonzero,
                "n_a_higher": n_a_higher,
                "n_b_higher": n_b_higher,
                "n_equal": n_equal,
            }
        )


    outcome_posthoc_df = pd.DataFrame(
        outcome_rows
    )


    # Holm correction across the 28 algorithm comparisons
    # within this outcome.
    rejected, corrected_p_values, _, _ = multipletests(
        outcome_posthoc_df[
            "p_uncorrected"
        ].to_numpy(),
        alpha=ALPHA,
        method="holm",
    )

    outcome_posthoc_df[
        "p_holm"
    ] = corrected_p_values

    outcome_posthoc_df[
        "significant_holm"
    ] = rejected

    all_posthoc_results.append(
        outcome_posthoc_df
    )


# -----------------------------------------------------------------------------
# Combine the current results
# -----------------------------------------------------------------------------

if all_posthoc_results:

    posthoc_results_df = (
        pd.concat(
            all_posthoc_results,
            ignore_index=True,
        )
        .sort_values(
            [
                "outcome",
                "p_holm",
                "p_uncorrected",
            ]
        )
        .reset_index(drop=True)
    )

    significant_posthoc_df = (
        posthoc_results_df.loc[
            posthoc_results_df[
                "significant_holm"
            ]
        ]
        .copy()
        .reset_index(drop=True)
    )

else:

    posthoc_results_df = pd.DataFrame()
    significant_posthoc_df = pd.DataFrame()


# -----------------------------------------------------------------------------
# Validate that all selected outcomes were processed
# -----------------------------------------------------------------------------

actual_posthoc_outcomes = (
    posthoc_results_df["outcome"]
    .drop_duplicates()
    .tolist()
    if not posthoc_results_df.empty
    else []
)

missing_posthoc_outcomes = [
    outcome
    for outcome in significant_outcomes
    if outcome not in actual_posthoc_outcomes
]

if missing_posthoc_outcomes:
    raise ValueError(
        "These selected outcomes were not added to "
        f"posthoc_results_df: {missing_posthoc_outcomes}"
    )


print(
    "Outcomes now present in posthoc_results_df:",
    actual_posthoc_outcomes
    if actual_posthoc_outcomes
    else "None",
)


# -----------------------------------------------------------------------------
# Display all pairwise comparisons
# -----------------------------------------------------------------------------

display_columns = [
    "outcome",
    "algorithm_a_label",
    "algorithm_b_label",
    "mean_a",
    "mean_b",
    "mean_difference_a_minus_b",
    "wilcoxon_w",
    "p_uncorrected",
    "p_holm",
    "rank_biserial_r",
    "direction",
    "n_pairs",
    "n_nonzero",
    "significant_holm",
]


if not posthoc_results_df.empty:

    display(
        posthoc_results_df[
            display_columns
        ].style.format(
            {
                "mean_a": "{:.2f}",
                "mean_b": "{:.2f}",
                "mean_difference_a_minus_b": "{:.2f}",
                "wilcoxon_w": "{:.2f}",
                "p_uncorrected": "{:.4f}",
                "p_holm": "{:.4f}",
                "rank_biserial_r": "{:.3f}",
            }
        )
    )


# -----------------------------------------------------------------------------
# Print post-hoc summary:
# uncorrected results and Holm-corrected results
# -----------------------------------------------------------------------------

print()
print("Post-hoc summary")
print("-" * 78)

if posthoc_results_df.empty:

    print(
        "No post-hoc comparisons were run because no "
        "Friedman outcome was significant."
    )

else:

    # Mark comparisons significant at the raw, uncorrected level.
    posthoc_results_df["significant_uncorrected"] = (
        posthoc_results_df["p_uncorrected"] < ALPHA
    )

    # Create separate result tables.
    significant_uncorrected_df = (
        posthoc_results_df.loc[
            posthoc_results_df[
                "significant_uncorrected"
            ]
        ]
        .copy()
        .reset_index(drop=True)
    )

    significant_holm_df = (
        posthoc_results_df.loc[
            posthoc_results_df[
                "significant_holm"
            ]
        ]
        .copy()
        .reset_index(drop=True)
    )

    summary_rows = []

    for outcome_name in significant_outcomes:

        outcome_results = (
            posthoc_results_df.loc[
                posthoc_results_df[
                    "outcome"
                ] == outcome_name
            ]
        )

        n_comparisons = len(
            outcome_results
        )

        n_significant_uncorrected = int(
            outcome_results[
                "significant_uncorrected"
            ].sum()
        )

        n_significant_holm = int(
            outcome_results[
                "significant_holm"
            ].sum()
        )

        summary_rows.append(
            {
                "outcome": outcome_name,
                "comparisons": n_comparisons,
                "significant_uncorrected": (
                    n_significant_uncorrected
                ),
                "significant_holm": (
                    n_significant_holm
                ),
            }
        )

        print(
            f"{outcome_name}: "
            f"{n_significant_uncorrected} of {n_comparisons} "
            "comparisons were significant without correction; "
            f"{n_significant_holm} of {n_comparisons} "
            "remained significant after Holm correction."
        )


    # -------------------------------------------------------------------------
    # Display compact summary table
    # -------------------------------------------------------------------------

    posthoc_summary_df = pd.DataFrame(
        summary_rows
    ).rename(
        columns={
            "outcome": "Outcome",
            "comparisons": "Comparisons",
            "significant_uncorrected": (
                "Significant without correction"
            ),
            "significant_holm": (
                "Significant after Holm"
            ),
        }
    )

    print()
    display(posthoc_summary_df)


    # -------------------------------------------------------------------------
    # Display significant comparisons without correction
    # -------------------------------------------------------------------------

    print()
    print("Significant comparisons without correction")
    print("-" * 78)

    if significant_uncorrected_df.empty:

        print(
            "No pairwise comparison had an uncorrected "
            "p-value below .05."
        )

    else:

        display(
            significant_uncorrected_df[
                display_columns
            ].style.format(
                {
                    "mean_a": "{:.2f}",
                    "mean_b": "{:.2f}",
                    "mean_difference_a_minus_b": "{:.2f}",
                    "wilcoxon_w": "{:.2f}",
                    "p_uncorrected": "{:.4f}",
                    "p_holm": "{:.4f}",
                    "rank_biserial_r": "{:.3f}",
                }
            )
        )


    # -------------------------------------------------------------------------
    # Display significant comparisons after Holm correction
    # -------------------------------------------------------------------------

    print()
    print("Significant comparisons after Holm correction")
    print("-" * 78)

    if significant_holm_df.empty:

        print(
            "No individual pairwise comparison remained "
            "significant after Holm correction."
        )

    else:

        display(
            significant_holm_df[
                display_columns
            ].style.format(
                {
                    "mean_a": "{:.2f}",
                    "mean_b": "{:.2f}",
                    "mean_difference_a_minus_b": "{:.2f}",
                    "wilcoxon_w": "{:.2f}",
                    "p_uncorrected": "{:.4f}",
                    "p_holm": "{:.4f}",
                    "rank_biserial_r": "{:.3f}",
                }
            )
        )


    print()
    print(
        "Interpretation: uncorrected results are exploratory. "
        "Formal pairwise conclusions should be based on the "
        "Holm-adjusted results."
    )


# -----------------------------------------------------------------------------
# Export complete post-hoc results
# -----------------------------------------------------------------------------

posthoc_output_path = (
    OUTPUT_DIR
    / "results_posthoc_wilcoxon.csv"
)

posthoc_results_df.to_csv(
    posthoc_output_path,
    index=False,
)

print()
print("Saved:", posthoc_output_path)

Outcomes selected for post-hoc testing: ['Viewing intention', 'Novelty']
Outcomes now present in posthoc_results_df: ['Novelty', 'Viewing intention']


,outcome,algorithm_a_label,algorithm_b_label,mean_a,mean_b,mean_difference_a_minus_b,wilcoxon_w,p_uncorrected,p_holm,rank_biserial_r,direction,n_pairs,n_nonzero,significant_holm
0,Novelty,Cast overlap,Weighted hybrid,1.42,1.04,0.38,0.00,0.0078,0.2188,1.000,Cast overlap higher,9,8,False
1,Novelty,KNN,Subtitles,0.84,1.69,-0.84,0.00,0.0156,0.4219,-1.000,Subtitles higher,9,7,False
2,Novelty,Cast overlap,KNN,1.42,0.84,0.58,3.00,0.0195,0.5078,0.889,Cast overlap higher,9,9,False
3,Novelty,KNN,TMDB,0.84,1.42,-0.58,3.00,0.0195,0.5078,-0.867,TMDB higher,9,9,False
4,Novelty,Subtitles,Weighted hybrid,1.69,1.04,0.64,4.00,0.0234,0.5625,0.822,Subtitles higher,9,9,False
5,Novelty,Image–text similarity,Subtitles,1.09,1.69,-0.60,4.50,0.0273,0.6289,-0.800,Subtitles higher,9,9,False
6,Novelty,Genome overlap,KNN,1.27,0.84,0.42,4.00,0.0312,0.6875,0.889,Genome overlap higher,9,9,False
7,Novelty,Image similarity,Subtitles,1.22,1.69,-0.47,2.50,0.0625,1.0000,-0.821,Subtitles higher,9,7,False
8,Novelty,Genome overlap,Subtitles,1.27,1.69,-0.42,7.50,0.0859,1.0000,-0.667,Subtitles higher,9,9,False
9,Novelty,Image similarity,KNN,1.22,0.84,0.38,4.00,0.1094,1.0000,0.750,Image similarity higher,9,7,False



Post-hoc summary
------------------------------------------------------------------------------
Viewing intention: 10 of 28 comparisons were significant without correction; 0 of 28 remained significant after Holm correction.
Novelty: 7 of 28 comparisons were significant without correction; 0 of 28 remained significant after Holm correction.



,Outcome,Comparisons,Significant without correction,Significant after Holm
0,Viewing intention,28,10,0
1,Novelty,28,7,0



Significant comparisons without correction
------------------------------------------------------------------------------


,outcome,algorithm_a_label,algorithm_b_label,mean_a,mean_b,mean_difference_a_minus_b,wilcoxon_w,p_uncorrected,p_holm,rank_biserial_r,direction,n_pairs,n_nonzero,significant_holm
0,Novelty,Cast overlap,Weighted hybrid,1.42,1.04,0.38,0.00,0.0078,0.2188,1.000,Cast overlap higher,9,8,False
1,Novelty,KNN,Subtitles,0.84,1.69,-0.84,0.00,0.0156,0.4219,-1.000,Subtitles higher,9,7,False
2,Novelty,Cast overlap,KNN,1.42,0.84,0.58,3.00,0.0195,0.5078,0.889,Cast overlap higher,9,9,False
3,Novelty,KNN,TMDB,0.84,1.42,-0.58,3.00,0.0195,0.5078,-0.867,TMDB higher,9,9,False
4,Novelty,Subtitles,Weighted hybrid,1.69,1.04,0.64,4.00,0.0234,0.5625,0.822,Subtitles higher,9,9,False
5,Novelty,Image–text similarity,Subtitles,1.09,1.69,-0.60,4.50,0.0273,0.6289,-0.800,Subtitles higher,9,9,False
6,Novelty,Genome overlap,KNN,1.27,0.84,0.42,4.00,0.0312,0.6875,0.889,Genome overlap higher,9,9,False
7,Viewing intention,KNN,Subtitles,3.89,2.42,1.47,0.00,0.0039,0.1094,1.000,KNN higher,9,9,False
8,Viewing intention,Cast overlap,Weighted hybrid,2.98,3.64,-0.67,0.00,0.0078,0.2109,-1.000,Weighted hybrid higher,9,8,False
9,Viewing intention,Cast overlap,KNN,2.98,3.89,-0.91,2.00,0.0117,0.3047,-0.933,KNN higher,9,9,False



Significant comparisons after Holm correction
------------------------------------------------------------------------------
No individual pairwise comparison remained significant after Holm correction.

Interpretation: uncorrected results are exploratory. Formal pairwise conclusions should be based on the Holm-adjusted results.

Saved: /app/evaluation_results/evaluation_outputs/results_posthoc_wilcoxon.csv


## Algorithm Rankings

The rankings below are descriptive decision-support summaries. They do not show
that one algorithm is statistically superior to another. All four criteria are
treated as benefit criteria, meaning that higher values receive better ranks.
This assumption is especially important for novelty: use the combined rankings
only when greater novelty is an intended objective.


### Individual Metric Rankings

Algorithms are ranked separately by relevance, intention, novelty, and Top-5
hit rate. Tied values receive the same minimum rank.


In [13]:
ranking_metrics = {
    "mean_relevance": "relevance_rank",
    "mean_intention": "intention_rank",
    "mean_novelty": "novelty_rank",
    "mean_top5_hit_rate": "top5_rank",
}

algorithm_rankings_df = algorithm_descriptives_df[
    ["algorithm", *ranking_metrics.keys()]
].copy()

for metric, rank_column in ranking_metrics.items():
    algorithm_rankings_df[rank_column] = (
        algorithm_rankings_df[metric]
        .rank(
            method="min",
            ascending=False,
        )
        .astype(int)
    )

algorithm_rankings_df["Algorithm"] = (
    algorithm_rankings_df["algorithm"]
    .map(algorithm_name_map)
    .fillna(
        algorithm_rankings_df["algorithm"]
        .str.replace("_", " ", regex=False)
        .str.title()
    )
)

individual_rankings_df = (
    algorithm_rankings_df[
        [
            "Algorithm",
            "relevance_rank",
            "intention_rank",
            "novelty_rank",
            "top5_rank",
        ]
    ]
    .sort_values(
        ["intention_rank", "relevance_rank", "Algorithm"]
    )
    .reset_index(drop=True)
)

display(individual_rankings_df)


,Algorithm,relevance_rank,intention_rank,novelty_rank,top5_rank
0,KNN,5,1,8,2
1,Weighted hybrid,1,2,7,2
2,Image--text similarity,2,3,6,1
3,Image similarity,4,4,5,4
4,Genome overlap,2,5,4,5
5,TMDB,7,6,2,7
6,Cast overlap,6,7,2,6
7,Subtitles,8,8,1,8


### Weighted Rankings and Reciprocal Rank Fusion

The weighted scores use min--max normalization across the eight algorithms.
Three scenarios are shown:

- **Equal:** 25% for each criterion.
- **Quality-focused:** 30% relevance, 30% Top-5 hit rate, 20% intention, and 20% novelty.
- **Discovery-focused:** 30% relevance, 20% Top-5 hit rate, 20% intention, and 30% novelty.

Reciprocal Rank Fusion (RRF) combines the four individual metric ranks with
$k=10$ and gives every metric equal importance.


In [14]:
# Normalize all ranking metrics to the interval [0, 1].
for metric in ranking_metrics:
    minimum = algorithm_rankings_df[metric].min()
    maximum = algorithm_rankings_df[metric].max()
    normalized_column = f"{metric}_normalized"

    if np.isclose(minimum, maximum):
        algorithm_rankings_df[normalized_column] = 1.0
    else:
        algorithm_rankings_df[normalized_column] = (
            algorithm_rankings_df[metric] - minimum
        ) / (maximum - minimum)


weight_sets = {
    "equal": {
        "mean_relevance": 0.25,
        "mean_top5_hit_rate": 0.25,
        "mean_intention": 0.25,
        "mean_novelty": 0.25,
    },
    "quality_focused": {
        "mean_relevance": 0.30,
        "mean_top5_hit_rate": 0.30,
        "mean_intention": 0.20,
        "mean_novelty": 0.20,
    },
    "discovery_focused": {
        "mean_relevance": 0.30,
        "mean_top5_hit_rate": 0.20,
        "mean_intention": 0.20,
        "mean_novelty": 0.30,
    },
}

for scenario, weights in weight_sets.items():
    score_column = f"weighted_score_{scenario}"
    rank_column = f"weighted_rank_{scenario}"

    algorithm_rankings_df[score_column] = 0.0

    for metric, weight in weights.items():
        algorithm_rankings_df[score_column] += (
            algorithm_rankings_df[f"{metric}_normalized"]
            * weight
        )

    algorithm_rankings_df[rank_column] = (
        algorithm_rankings_df[score_column]
        .rank(
            method="min",
            ascending=False,
        )
        .astype(int)
    )


# Reciprocal Rank Fusion using the four individual metric ranks.
RRF_K = 10
individual_rank_columns = list(ranking_metrics.values())

algorithm_rankings_df["rrf_score"] = sum(
    1 / (RRF_K + algorithm_rankings_df[rank_column])
    for rank_column in individual_rank_columns
)

algorithm_rankings_df["rrf_rank"] = (
    algorithm_rankings_df["rrf_score"]
    .rank(
        method="min",
        ascending=False,
    )
    .astype(int)
)

algorithm_rankings_df = (
    algorithm_rankings_df
    .sort_values(
        ["rrf_rank", "weighted_rank_equal", "Algorithm"]
    )
    .reset_index(drop=True)
)

# Save the complete numeric ranking results.
ranking_output_path = (
    OUTPUT_DIR / "results_algorithm_rankings.csv"
)
algorithm_rankings_df.to_csv(
    ranking_output_path,
    index=False,
)

# Compact report table containing ranks only.
ranking_report_df = algorithm_rankings_df[
    [
        "Algorithm",
        "relevance_rank",
        "intention_rank",
        "novelty_rank",
        "top5_rank",
        "weighted_rank_equal",
        "weighted_rank_quality_focused",
        "weighted_rank_discovery_focused",
        "rrf_rank",
    ]
].rename(
    columns={
        "relevance_rank": "Rel.",
        "intention_rank": "Pref.",
        "novelty_rank": "Nov.",
        "top5_rank": "Top-5",
        "weighted_rank_equal": "Equal",
        "weighted_rank_quality_focused": "Quality",
        "weighted_rank_discovery_focused": "Discovery",
        "rrf_rank": "RRF",
    }
)

display(ranking_report_df)

ranking_latex_code = ranking_report_df.to_latex(
    index=False,
    escape=False,
    caption=(
        "Descriptive algorithm ranks by individual metrics, weighted "
        "scenarios, and reciprocal rank fusion. Lower ranks are better."
    ),
    label="tab:algorithm-rankings",
    position="htbp",
    column_format="lcccccccc",
)

ranking_latex_code = ranking_latex_code.replace(
    "\\begin{table}[htbp]\n",
    "\\begin{table}[htbp]\n\\centering\n\\small\n",
)

ranking_latex_path = (
    OUTPUT_DIR / "table_algorithm_rankings.tex"
)
ranking_latex_path.write_text(
    ranking_latex_code,
    encoding="utf-8",
)

print("Saved:", ranking_output_path)
print("Saved:", ranking_latex_path)
print()
print(ranking_latex_code)


,Algorithm,Rel.,Pref.,Nov.,Top-5,Equal,Quality,Discovery,RRF
0,Weighted hybrid,1,2,7,2,1,1,1,1
1,Image--text similarity,2,3,6,1,2,2,2,2
2,KNN,5,1,8,2,4,4,5,3
3,Genome overlap,2,5,4,5,5,5,4,4
4,Image similarity,4,4,5,4,3,3,3,5
5,Cast overlap,6,7,2,6,6,6,6,6
6,TMDB,7,6,2,7,7,7,7,7
7,Subtitles,8,8,1,8,8,8,8,8


Saved: /app/evaluation_results/evaluation_outputs/results_algorithm_rankings.csv
Saved: /app/evaluation_results/evaluation_outputs/table_algorithm_rankings.tex

\begin{table}[htbp]
\centering
\small
\caption{Descriptive algorithm ranks by individual metrics, weighted scenarios, and reciprocal rank fusion. Lower ranks are better.}
\label{tab:algorithm-rankings}
\begin{tabular}{lcccccccc}
\toprule
Algorithm & Rel. & Pref. & Nov. & Top-5 & Equal & Quality & Discovery & RRF \\
\midrule
Weighted hybrid & 1 & 2 & 7 & 2 & 1 & 1 & 1 & 1 \\
Image--text similarity & 2 & 3 & 6 & 1 & 2 & 2 & 2 & 2 \\
KNN & 5 & 1 & 8 & 2 & 4 & 4 & 5 & 3 \\
Genome overlap & 2 & 5 & 4 & 5 & 5 & 5 & 4 & 4 \\
Image similarity & 4 & 4 & 5 & 4 & 3 & 3 & 3 & 5 \\
Cast overlap & 6 & 7 & 2 & 6 & 6 & 6 & 6 & 6 \\
TMDB & 7 & 6 & 2 & 7 & 7 & 7 & 7 & 7 \\
Subtitles & 8 & 8 & 1 & 8 & 8 & 8 & 8 & 8 \\
\bottomrule
\end{tabular}
\end{table}



In [15]:
# =============================================================================
# TOP 3 ALGORITHMS FOR EACH RANKING
# =============================================================================

# Ranking column and corresponding score/metric used to resolve ties
# consistently for presentation.
ranking_definitions = {
    "Relevance": {
        "rank": "relevance_rank",
        "score": "mean_relevance",
    },
    "Viewing intention": {
        "rank": "intention_rank",
        "score": "mean_intention",
    },
    "Novelty": {
        "rank": "novelty_rank",
        "score": "mean_novelty",
    },
    "Top-5 hit rate": {
        "rank": "top5_rank",
        "score": "mean_top5_hit_rate",
    },
    "Equal weights": {
        "rank": "weighted_rank_equal",
        "score": "weighted_score_equal",
    },
    "Quality focused": {
        "rank": "weighted_rank_quality_focused",
        "score": "weighted_score_quality_focused",
    },
    "Discovery focused": {
        "rank": "weighted_rank_discovery_focused",
        "score": "weighted_score_discovery_focused",
    },
    "Reciprocal Rank Fusion": {
        "rank": "rrf_rank",
        "score": "rrf_score",
    },
}


top_3_rows = []

for ranking_name, columns in ranking_definitions.items():
    rank_column = columns["rank"]
    score_column = columns["score"]

    ranking_top_3 = (
        algorithm_rankings_df
        .sort_values(
            [rank_column, score_column, "Algorithm"],
            ascending=[True, False, True],
        )
        .head(3)
        .reset_index(drop=True)
    )

    top_3_rows.append(
        {
            "Ranking": ranking_name,
            "1st": (
                f"{ranking_top_3.loc[0, 'Algorithm']} "
                f"(rank {int(ranking_top_3.loc[0, rank_column])})"
            ),
            "2nd": (
                f"{ranking_top_3.loc[1, 'Algorithm']} "
                f"(rank {int(ranking_top_3.loc[1, rank_column])})"
            ),
            "3rd": (
                f"{ranking_top_3.loc[2, 'Algorithm']} "
                f"(rank {int(ranking_top_3.loc[2, rank_column])})"
            ),
        }
    )


top_3_rankings_df = pd.DataFrame(top_3_rows)

display(top_3_rankings_df)


# -----------------------------------------------------------------------------
# Save as CSV
# -----------------------------------------------------------------------------

top_3_csv_path = (
    OUTPUT_DIR
    / "results_algorithm_rankings_top3.csv"
)

top_3_rankings_df.to_csv(
    top_3_csv_path,
    index=False,
)


# -----------------------------------------------------------------------------
# Export as LaTeX
# -----------------------------------------------------------------------------

top_3_latex_code = top_3_rankings_df.to_latex(
    index=False,
    escape=False,
    caption=(
        "Top three recommendation algorithms under each "
        "individual and combined ranking method."
    ),
    label="tab:algorithm-rankings-top3",
    position="htbp",
    column_format="llll",
)

top_3_latex_code = top_3_latex_code.replace(
    "\\begin{table}[htbp]\n",
    "\\begin{table}[htbp]\n\\centering\n\\small\n",
)

top_3_latex_path = (
    OUTPUT_DIR
    / "table_algorithm_rankings_top3.tex"
)

top_3_latex_path.write_text(
    top_3_latex_code,
    encoding="utf-8",
)


print("Saved:", top_3_csv_path)
print("Saved:", top_3_latex_path)
print()
print(top_3_latex_code)

,Ranking,1st,2nd,3rd
0,Relevance,Weighted hybrid (rank 1),Genome overlap (rank 2),Image--text similarity (rank 2)
1,Viewing intention,KNN (rank 1),Weighted hybrid (rank 2),Image--text similarity (rank 3)
2,Novelty,Subtitles (rank 1),Cast overlap (rank 2),TMDB (rank 2)
3,Top-5 hit rate,Image--text similarity (rank 1),KNN (rank 2),Weighted hybrid (rank 2)
4,Equal weights,Weighted hybrid (rank 1),Image--text similarity (rank 2),Image similarity (rank 3)
5,Quality focused,Weighted hybrid (rank 1),Image--text similarity (rank 2),Image similarity (rank 3)
6,Discovery focused,Weighted hybrid (rank 1),Image--text similarity (rank 2),Image similarity (rank 3)
7,Reciprocal Rank Fusion,Weighted hybrid (rank 1),Image--text similarity (rank 2),KNN (rank 3)


Saved: /app/evaluation_results/evaluation_outputs/results_algorithm_rankings_top3.csv
Saved: /app/evaluation_results/evaluation_outputs/table_algorithm_rankings_top3.tex

\begin{table}[htbp]
\centering
\small
\caption{Top three recommendation algorithms under each individual and combined ranking method.}
\label{tab:algorithm-rankings-top3}
\begin{tabular}{llll}
\toprule
Ranking & 1st & 2nd & 3rd \\
\midrule
Relevance & Weighted hybrid (rank 1) & Genome overlap (rank 2) & Image--text similarity (rank 2) \\
Viewing intention & KNN (rank 1) & Weighted hybrid (rank 2) & Image--text similarity (rank 3) \\
Novelty & Subtitles (rank 1) & Cast overlap (rank 2) & TMDB (rank 2) \\
Top-5 hit rate & Image--text similarity (rank 1) & KNN (rank 2) & Weighted hybrid (rank 2) \\
Equal weights & Weighted hybrid (rank 1) & Image--text similarity (rank 2) & Image similarity (rank 3) \\
Quality focused & Weighted hybrid (rank 1) & Image--text similarity (rank 2) & Image similarity (rank 3) \\
Discovery fo